# 15 SFT_And_Alignment_Basics

## Problem

为什么一个预训练模型还不等于一个好用的对话助手？SFT 在整个对齐流程里处于什么位置，它到底改变了什么？

## Dependencies

建议先理解预训练目标、teacher forcing、token-level loss 和 prompt-response 数据。


## Module_Goals

- 理解 instruction tuning / SFT 的核心目标
- 理解 prompt-response 数据格式为什么重要
- 理解 SFT 和预训练在 loss 形式上的联系与差异
- 理解 SFT 在多阶段训练链路里的职责

## Scope_And_Approach

这个 notebook 重点说明 SFT 为什么是一个单独阶段。它和预训练看起来都在做 token-level loss，但它们优化的分布不同：预训练优化广义文本分布，SFT 优化更接近任务和交互目标的回答分布。


## Context_And_Motivation

预训练模型学到的是“语言世界里什么常常跟着什么出现”。这很强，但它并不天然等于“会按人类预期回答问题”。

SFT 做的事，是用更明确的指令-回答样本告诉模型：

- 当用户这样提问时，什么样的回答结构更合适。
- 哪些内容应该优先出现。
- 什么语气、格式、步骤或约束更符合任务要求。

所以 SFT 不是重学语言，而是在已有语言能力之上，把输出分布重新往“更像一个任务助手”的方向拉过去。

从多阶段训练视角看，SFT 的位置非常关键：

- pretraining 给能力底座
- continued training 给某些能力补强
- SFT 给行为骨架
- 后面的 reward model 与 RLHF 再继续对行为做偏好层面的细调


In [ ]:
# 一个最小 SFT 样本
sample = {
    'prompt': '用户：请用一句话解释什么是 BPE。',
    'response': '助手：BPE 是一种通过反复合并高频符号对来构建子词词表的分词方法。'
}

formatted_text = sample['prompt'] + '\n' + sample['response']
print(formatted_text)


In [ ]:
import numpy as np

# 这里只展示 loss mask 思路：用户部分不计损失，助手部分计损失
loss_mask = np.array([0, 0, 0, 1, 1, 1, 1, 1])
logits = np.random.randn(8, 10)
targets = np.array([1, 2, 3, 4, 5, 6, 7, 8])

def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

probs = softmax(logits)
per_token_loss = -np.log(probs[np.arange(len(targets)), targets])
masked_loss = (per_token_loss * loss_mask).sum() / loss_mask.sum()

print('per_token_loss =', per_token_loss)
print('loss_mask =', loss_mask)
print('masked_loss =', masked_loss)


In [ ]:
# 一个最小阶段流转示意
stage_shift = {
    'pretraining_output': 'general next-token predictor',
    'sft_output': 'instruction-following assistant skeleton',
    'rlhf_output': 'preference-shaped assistant behavior',
}

for key, value in stage_shift.items():
    print(f'{key}: {value}')


## Math_And_Data_View

SFT 和预训练在表面形式上都可以写成 token-level cross entropy，但它们的训练数据分布不同：

- 预训练：广义自然文本、代码、文档、知识语料
- SFT：更显式的指令-回答、格式约束、任务示范数据

也就是说，SFT 并不是换了一个完全不同的 loss，而是换了一个更强任务导向的数据分布。

## Trade_Offs_And_Risk_Points

- 把 SFT 想成和预训练完全不同的机器。很多时候它们共享相似的 token-level loss 形式。
- 忽略数据格式，认为只要有问答文本就行。实际格式会显著影响模型学到的对话习惯。
- 没有区分“语言能力”与“按指令配合”。前者主要来自预训练，后者更多来自 SFT 和后续对齐。
- 误以为 SFT 足以完成全部行为对齐。SFT 给的是行为骨架，不一定足以覆盖偏好细节和长期稳定性。

## Checkpoints

- 自己设计一个多轮对话格式样本。
- 思考为什么很多实现只对 assistant 部分计算 loss。
- 解释 SFT 为什么通常能显著改善模型的可用性。
- 用自己的话说明 SFT 在 pretraining 和 RLHF 之间起什么作用。

## Summary

SFT 让模型从“会续写文本”转向“更会参与任务式交互”。它不负责建立全部知识底座，也不负责完成全部偏好塑形，但它是把通用模型转成助手型模型的关键中间层。

## Next_Module

下一步进入奖励模型和偏好优化。到那里，重点会从“会回答”继续转向“更符合偏好地回答”。
